In [1]:
from pathlib import Path
from PIL import Image, ImageFilter
import numpy as np
import csv

# Folder containing your five original figures
folder = Path(
    r"D:\MSC THESIS AAVAS\AFTER MID DEFENSE\FINAL OUTPUT\5 figures"
)

# Folder for improved figures
output_folder = folder / "Improved_Figures"
output_folder.mkdir(parents=True, exist_ok=True)

print("Folder:", folder)
print("Folder exists:", folder.exists())

Folder: D:\MSC THESIS AAVAS\AFTER MID DEFENSE\FINAL OUTPUT\5 figures
Folder exists: True


In [2]:
%pip install pillow numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Supported image formats
extensions = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

# Find all images
image_files = sorted(
    file for file in folder.iterdir()
    if file.is_file() and file.suffix.lower() in extensions
)

print(f"Total images found: {len(image_files)}\n")

for i, file in enumerate(image_files, start=1):

    with Image.open(file) as img:

        width, height = img.size

        aspect_ratio = width / height

        file_size = file.stat().st_size / (1024 * 1024)

        print(f"Figure {i}")
        print(f"Filename: {file.name}")
        print(f"Dimensions: {width} × {height} pixels")
        print(f"Aspect ratio: {aspect_ratio:.2f}")
        print(f"File size: {file_size:.2f} MB")
        print("-" * 50)

Total images found: 6

Figure 1
Filename: 7a1a656e-4c35-4deb-a899-a6922702232a.png
Dimensions: 1405 × 1120 pixels
Aspect ratio: 1.25
File size: 1.29 MB
--------------------------------------------------
Figure 2
Filename: Actual_vs_Predicted_Tuned_Random_Forest.png
Dimensions: 1655 × 1396 pixels
Aspect ratio: 1.19
File size: 0.11 MB
--------------------------------------------------
Figure 3
Filename: bootstrap_oob.png
Dimensions: 2800 × 900 pixels
Aspect ratio: 3.11
File size: 0.11 MB
--------------------------------------------------
Figure 4
Filename: Feature_Importance_Tuned_Random_Forest.png
Dimensions: 2283 × 1258 pixels
Aspect ratio: 1.81
File size: 0.07 MB
--------------------------------------------------
Figure 5
Filename: Figure_01_Correlation_Matrix.png
Dimensions: 2490 × 1985 pixels
Aspect ratio: 1.25
File size: 0.24 MB
--------------------------------------------------
Figure 6
Filename: Figure_06_Combined_RF_Partial_Dependence.png
Dimensions: 3613 × 2342 pixels
Aspect ra

In [4]:
def remove_white_margins(img, padding=20):

    img = img.convert("RGB")

    arr = np.asarray(img)

    # Identify pixels that are not nearly white
    content_mask = np.any(arr < 245, axis=2)

    rows, cols = np.where(content_mask)

    if len(rows) == 0:
        return img

    left = max(0, cols.min() - padding)
    right = min(img.width, cols.max() + padding + 1)

    top = max(0, rows.min() - padding)
    bottom = min(img.height, rows.max() + padding + 1)

    cropped = img.crop((left, top, right, bottom))

    return cropped

In [5]:
def reduce_title_gap(img):

    img = img.convert("RGB")

    arr = np.asarray(img)

    width, height = img.size

    # Count non-white pixels in each row
    ink_count = np.any(arr < 245, axis=2).sum(axis=1)

    # Rows containing almost no visible content
    threshold = max(2, int(width * 0.001))

    blank = ink_count <= threshold

    # Search for large blank bands near the top
    start = None
    candidates = []

    for y in range(height):

        if blank[y] and start is None:
            start = y

        elif not blank[y] and start is not None:

            end = y

            if (
                start > 10
                and end < height * 0.40
                and end - start > height * 0.05
            ):
                candidates.append((start, end))

            start = None

    if not candidates:
        return img

    # Select the largest blank band
    start, end = max(
        candidates,
        key=lambda x: x[1] - x[0]
    )

    # Retain some whitespace for readability
    keep = 12

    if end - start <= 2 * keep:
        return img

    top_part = img.crop(
        (0, 0, width, start + keep)
    )

    bottom_part = img.crop(
        (0, end - keep, width, height)
    )

    new_height = top_part.height + bottom_part.height

    result = Image.new(
        "RGB",
        (width, new_height),
        "white"
    )

    result.paste(top_part, (0, 0))
    result.paste(bottom_part, (0, top_part.height))

    return result

In [6]:
def identify_figure(filename):
    """
    Identify the figure type and recommended
    layout based on its filename.
    """

    name = filename.lower()

    if "correlation" in name:
        return "Correlation Matrix", "single"

    elif "actual" in name and "predicted" in name:
        return "Actual vs Predicted", "single"

    elif "importance" in name:
        return "Feature Importance", "single"

    elif "partial" in name or "dependence" in name:
        return "Partial Dependence", "double"

    elif "bootstrap" in name or "oob" in name:
        return "Bootstrap OOB", "double"

    else:
        return "Unknown Figure", "single"


print("Figure identification function loaded successfully!")

Figure identification function loaded successfully!


In [7]:
def improve_figure(file, output_folder):

    figure_type, layout = identify_figure(file.name)

    img = Image.open(file).convert("RGB")

    original_width, original_height = img.size

    # Remove excessive outer margins
    img = remove_white_margins(
        img,
        padding=20
    )

    # Special adjustment for partial dependence figure
    if figure_type == "Partial Dependence":

        img = reduce_title_gap(img)

        img = remove_white_margins(
            img,
            padding=15
        )

    # Target pixel widths for publication
    if layout == "single":
        target_width = 1400

    else:
        target_width = 2600

    # Upscale only if needed
    scale = min(
        2.0,
        max(1.0, target_width / img.width)
    )

    if scale > 1.01:

        new_width = int(img.width * scale)
        new_height = int(img.height * scale)

        img = img.resize(
            (new_width, new_height),
            Image.Resampling.LANCZOS
        )

    # Gentle sharpening
    img = img.filter(
        ImageFilter.UnsharpMask(
            radius=1.0,
            percent=105,
            threshold=3
        )
    )

    # Output filename
    output_name = f"Improved_{file.stem}.png"

    output_path = output_folder / output_name

    # Save publication-quality PNG
    img.save(
        output_path,
        format="PNG",
        dpi=(300, 300),
        optimize=True
    )

    print(f"\nFigure: {figure_type}")

    print(
        f"Original: "
        f"{original_width} × {original_height}"
    )

    print(
        f"Improved: "
        f"{img.width} × {img.height}"
    )

    print(f"Saved: {output_path}")

    return {
        "figure": figure_type,
        "original_file": file.name,
        "output_file": output_name,
        "layout": layout,
        "original_size": f"{original_width}x{original_height}",
        "improved_size": f"{img.width}x{img.height}"
    }


# PROCESS ALL FIGURES

reports = []

for file in image_files:

    report = improve_figure(
        file,
        output_folder
    )

    reports.append(report)

print("\nAll figures processed successfully.")


Figure: Unknown Figure
Original: 1405 × 1120
Improved: 1405 × 1079
Saved: D:\MSC THESIS AAVAS\AFTER MID DEFENSE\FINAL OUTPUT\5 figures\Improved_Figures\Improved_7a1a656e-4c35-4deb-a899-a6922702232a.png

Figure: Actual vs Predicted
Original: 1655 × 1396
Improved: 1640 × 1377
Saved: D:\MSC THESIS AAVAS\AFTER MID DEFENSE\FINAL OUTPUT\5 figures\Improved_Figures\Improved_Actual_vs_Predicted_Tuned_Random_Forest.png

Figure: Bootstrap OOB
Original: 2800 × 900
Improved: 2653 × 871
Saved: D:\MSC THESIS AAVAS\AFTER MID DEFENSE\FINAL OUTPUT\5 figures\Improved_Figures\Improved_bootstrap_oob.png

Figure: Feature Importance
Original: 2283 × 1258
Improved: 2265 × 1238
Saved: D:\MSC THESIS AAVAS\AFTER MID DEFENSE\FINAL OUTPUT\5 figures\Improved_Figures\Improved_Feature_Importance_Tuned_Random_Forest.png

Figure: Correlation Matrix
Original: 2490 × 1985
Improved: 2468 × 1959
Saved: D:\MSC THESIS AAVAS\AFTER MID DEFENSE\FINAL OUTPUT\5 figures\Improved_Figures\Improved_Figure_01_Correlation_Matrix.png



In [8]:
import matplotlib.pyplot as plt

fig = plt.gcf()

for ax in fig.axes:
    ax.set_ylabel(
        r'Average Predicted $V_{85}$ (km/h)',
        fontsize=12
    )

    ax.tick_params(
        axis='both',
        labelsize=11
    )

fig.tight_layout()

fig.savefig(
    "Improved_RF_Partial_Dependence.png",
    dpi=400,
    bbox_inches="tight"
)

plt.show()

<Figure size 640x480 with 0 Axes>